<a href="https://colab.research.google.com/github/Arturgds/ProjetoFinalIA/blob/main/ProjetoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SETUP

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from packaging import version
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

assert sys.version_info >= (3, 8)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)
warnings.filterwarnings("ignore")

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize=12)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)
sns.set_style("whitegrid")

IMAGES_PATH = Path("images/classification_project")
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution, bbox_inches="tight")

# Projeto Final de Aprendizado de Máquina

## 5.1 Identificação e descrição do problema

### Título
Previsão do tipo de assinatura de usuários em uma plataforma de streaming musical.

### Integrantes
- Artur Gabriel Dantas Santos.
- Kaíque Teixeira Rodrigues.
- Matheus Mendonca Santana.

### Fonte dos dados
Conjunto de dados **Music Streaming Habits 2026**, disponível publicamente no Kaggle.

### Objetivo
Desenvolver um modelo de aprendizado de máquina capaz de prever o tipo de assinatura de um usuário com base em seus hábitos de consumo musical.

### Atributo-alvo
O atributo-alvo definido para este projeto é **`subscription`**, que representa o tipo de assinatura do usuário, com as classes:
- Free
- Premium
- Family
- Student

### Atributos preditivos
Os atributos utilizados como variáveis preditoras são:
- `age`
- `country`
- `platform`
- `top_genre`
- `top_artist`
- `daily_listening_minutes`
- `songs_per_day`
- `playlists_count`
- `skip_rate_pct`
- `discover_weekly_user`
- `top_mood`
- `uses_offline_mode`
- `podcasts_too`

O atributo `listener_id` não será utilizado na modelagem, pois funciona apenas como identificador do registro.

### Tipo da tarefa
O problema foi formulado como uma tarefa de **classificação multiclasse**, pois o atributo-alvo `subscription` possui mais de uma categoria possível e não representa um valor numérico contínuo.

## 5.2 Compreensão dos dados

Nesta etapa, foi feita uma leitura inicial do conjunto de dados para entender sua estrutura, qualidade e possíveis necessidades de tratamento antes da modelagem.

### Quantidade de registros e atributos
O conjunto de dados possui **4000 registros** e **15 colunas**. A coluna `listener_id` funciona apenas como identificador e não será usada como variável preditiva, então o problema será modelado com **14 atributos úteis**.

### Tipos das variáveis
O dataset contém variáveis numéricas, categóricas e booleanas. As colunas numéricas são `age`, `daily_listening_minutes`, `songs_per_day`, `playlists_count` e `skip_rate_pct`. As colunas categóricas são `country`, `platform`, `subscription`, `top_genre`, `top_artist` e `top_mood`. As colunas booleanas são `discover_weekly_user`, `uses_offline_mode` e `podcasts_too`.

### Valores ausentes
A verificação de valores ausentes mostrou **0 valores nulos no total**. Isso indica que não será necessário imputar valores ausentes nesta base.

### Duplicações
Foram encontrados **0 registros duplicados**. Não há duplicações relevantes, então os dados podem seguir para a análise e modelagem.

### Inconsistências
Em uma verificação inicial, não foram observadas inconsistências óbvias nas colunas principais, como valores incompatíveis com o tipo esperado ou categorias fora do padrão. Ainda assim, essa etapa deve ser complementada com análises específicas nas variáveis categóricas e na faixa de valores das variáveis numéricas.

### Distribuição do atributo-alvo
A variável-alvo `subscription` apresenta a seguinte distribuição:
- **Free**: 1820 registros (45.5%)
- **Student**: 401 registros (10.02%)
- **Premium**: 1193 registros (29.83%)
- **Family**: 596 registros (14.65%)

### Desbalanceamento
Há sinais de desbalanceamento moderado entre as classes. Como o problema é de classificação multiclasse, essa distribuição precisa ser considerada na divisão treino/teste e na escolha das métricas de avaliação. A acurácia sozinha não deve ser usada como única referência de desempenho; por isso, também serão observados matriz de confusão, precisão, revocação e F1-score.